# Feature Engineering Using Scikit-Learn

### Do You Want to Build a Snowman?

Prepare the inputs for a model that predicts snowman height.

<img src="./assets/olaf.jpeg" width="300" alt="Snowman illustration">

This deliberately tiny, fictional dataset is a **preprocessing exercise**, not evidence that lunch or dinner predicts snowman height. Treat all rows as a supplied training sample; do not report a model generalization score from these 14 examples.

**Business connection:** in a housing dataset, the same steps could impute missing floor areas, scale numerical measurements, and encode property types before predicting price. The workflow transfers; the useful features and preprocessing choices depend on the task.

#### Load Packages

In [1]:
import numpy as np
import pandas as pd

#### Load Data

In [2]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data).fillna(np.nan)
df_train

,temp,lunch,dinner,precipitation,height_snowman_cm
0,-3.0,soup,pizza,yes,100
1,5.0,sandwich,pizza,no,0
2,0.0,soup,noodles,yes,75
3,7.0,burger,NaN,yes,0
4,3.0,sandwich,fishsticks,yes,20
5,-1.0,soup,pizza,yes,25
6,1.0,cereal,NaN,no,0
7,NaN,salad,fishsticks,yes,35
8,-6.0,sandwich,noodles,yes,170
9,3.0,burger,pizza,no,0


## Exercise

1. Separate `df_train` into `X_train` and `y_train`; the target is `height_snowman_cm`.
2. Identify numeric, nominal, and binary inputs, then inspect missing values.
3. Choose and justify an imputation strategy for each incomplete input. Row order has no time meaning here, so interpolation between neighboring rows is not justified.
4. Choose suitable encodings and a scaler. Explain which columns need each transformation and avoid inventing an order for nominal categories.
5. Import the scikit-learn tools you choose, apply them, and combine the transformed columns into a DataFrame.
6. Produce `X_train_fe` with numerical values, no missing values, the same row count and index as `X_train`, and no target column.

Fit on training data only. To transform new rows later, reuse the fitted objects instead of recomputing statistics.

In [3]:
target = "height_snowman_cm"
X_train = df_train.drop(target, axis=1)
y_train = df_train[target]

In [4]:
X_train.isna().sum()

temp             2
lunch            0
dinner           3
precipitation    0
dtype: int64

In [5]:
from sklearn.impute import SimpleImputer

temp_imputer = SimpleImputer(strategy="median").set_output(transform="pandas")
temp_imputer.fit(X_train[["temp"]])
temp_imputed_train = temp_imputer.transform(X_train[["temp"]])
temp_imputed_train

,temp
0,-3.0
1,5.0
2,0.0
3,7.0
4,3.0
5,-1.0
6,1.0
7,0.0
8,-6.0
9,3.0


In [6]:
from sklearn.preprocessing import StandardScaler

temp_scaler = StandardScaler().set_output(transform="pandas")
temp_scaled_train = temp_scaler.fit_transform(temp_imputed_train)
temp_scaled_train

,temp
0,-1.078720
1,1.438293
2,-0.134840
3,2.067546
4,0.809040
5,-0.449467
6,0.179787
7,-0.134840
8,-2.022600
9,0.809040


In [7]:
dinner_imputer = SimpleImputer(strategy="most_frequent").set_output(transform="pandas")
dinner_imputer.fit(X_train[["dinner"]])
dinner_imputed_train = dinner_imputer.transform(X_train[["dinner"]])
dinner_imputed_train

,dinner
0,pizza
1,pizza
2,noodles
3,pizza
4,fishsticks
5,pizza
6,pizza
7,fishsticks
8,noodles
9,pizza


In [8]:
from sklearn.preprocessing import OneHotEncoder

subset_categoric = ["lunch", "precipitation"]

ohe_encoder = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
lunch_prep_encoded_train = ohe_encoder.fit_transform(X_train[subset_categoric])

dinner_encoder = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
dinner_encoded_train = dinner_encoder.fit_transform(dinner_imputed_train)

In [9]:
X_train_fe = pd.concat(
    [temp_scaled_train, dinner_encoded_train, lunch_prep_encoded_train], axis=1
)
X_train_fe

,temp,dinner_noodles,dinner_pizza,lunch_cereal,lunch_salad,lunch_sandwich,lunch_soup,precipitation_yes
0,-1.078720,0.0,1.0,0.0,0.0,0.0,1.0,1.0
1,1.438293,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,-0.134840,1.0,0.0,0.0,0.0,0.0,1.0,1.0
3,2.067546,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,0.809040,0.0,0.0,0.0,0.0,1.0,0.0,1.0
5,-0.449467,0.0,1.0,0.0,0.0,0.0,1.0,1.0
6,0.179787,0.0,1.0,1.0,0.0,0.0,0.0,0.0
7,-0.134840,0.0,0.0,0.0,1.0,0.0,0.0,1.0
8,-2.022600,1.0,0.0,0.0,0.0,1.0,0.0,1.0
9,0.809040,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [10]:
assert X_train_fe.index.equals(y_train.index)
assert not X_train_fe.isna().any().any()

### Check Your Transformations on New Inputs

Reuse your fitted objects to transform the two rows below, **without calling `fit` or `fit_transform` again**. Keep the training output columns and the new row indices, and check that no missing values remain. `wrap` is an unseen lunch category: explain whether your encoder raises an error or handles it explicitly. An expected error is a valid observation; do not silently refit on these rows.

In [11]:
# New inputs only: the fitted transformers must not learn from these rows.
X_new = pd.DataFrame(
    {
        "temp": [np.nan, 2.0],
        "lunch": ["soup", "wrap"],
        "dinner": [np.nan, "pizza"],
        "precipitation": ["yes", "no"],
    },
    index=["known", "new_category"],
)
X_new

,temp,lunch,dinner,precipitation
known,NaN,soup,NaN,yes
new_category,2.0,wrap,pizza,no


In [12]:
ohe_encoder.transform(X_new.loc[["new_category"], subset_categoric])

ValueError: Found unknown categories ['wrap'] in column 0 during transform

In [13]:
known = X_new.loc[["known"]]
known_temp = temp_scaler.transform(temp_imputer.transform(known[["temp"]]))
known_dinner = dinner_encoder.transform(dinner_imputer.transform(known[["dinner"]]))
known_other = ohe_encoder.transform(known[subset_categoric])
known_fe = pd.concat([known_temp, known_dinner, known_other], axis=1)
print(known_fe)

try:
    ohe_encoder.transform(X_new.loc[["new_category"], subset_categoric])
except ValueError as error:
    print(f"Expected unseen-category error: {error}")

          temp  dinner_noodles  dinner_pizza  lunch_cereal  lunch_salad  \
known -0.13484             0.0           1.0           0.0          0.0   

       lunch_sandwich  lunch_soup  precipitation_yes  
known             0.0         1.0                1.0  
Expected unseen-category error: Found unknown categories ['wrap'] in column 0 during transform


### Check Your Understanding

1. Why should you use `transform`, not `fit_transform`, on the test set?
2. Why could encoding meals as 0, 1, 2 mislead a linear or distance-based model?
3. Does `RobustScaler` remove extreme observations? Explain briefly.

Compare your reasoning with the short answers at the end of the solution.